# Evaluate Fine tune Phi3-mini

## Enviroments



In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install peft transformers evaluate rouge-score bert-score datasets accelerate bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from evaluate import load
from bert_score import score
import json
from datasets import load_dataset
from unsloth import FastLanguageModel

## Load dataset

In [2]:
from datasets import load_dataset
dataset = load_dataset("Hoang42/summary_article")

README.md:   0%|          | 0.00/197 [00:00<?, ?B/s]

data/train.jsonl:   0%|          | 0.00/716M [00:00<?, ?B/s]

data/validation.jsonl:   0%|          | 0.00/89.2M [00:00<?, ?B/s]

data/test.jsonl:   0%|          | 0.00/89.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/220520 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/27565 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/27565 [00:00<?, ? examples/s]

In [3]:
val = dataset["validation"]
new_eval = val.shuffle(seed=42).select(range(50))


In [4]:
for sample in new_eval:
  print(sample['input'])
  print('###Ouput:', sample['output'])

Tiêu đề: Hải Phòng sẽ kiểm tra, xử phạt khách du lịch vi phạm quy định phòng dịch

Nội dung: Vui mừng đón khách du lịch trong trạng thái bình thường mới Theo chỉ đạo của UBND TP.Hải Phòng, từ ngày 1.10, thành phố cho phép các địa phương mở các khu, điểm du lịch, danh lam thắng cảnh nhưng chỉ đón và phục vụ khách nội tỉnh. Khách và người trực tiếp hướng dẫn khách đến tham quan phải đảm bảo một trong các điều kiện: Có kết quả xét nghiệm âm tính với SARS-CoV-2 bằng phương pháp Test nhanh kháng nguyên hoặc RT-PCR trong vòng 72 giờ; có chứng nhận tiêm đủ 2 mũi vaccine phòng COVID-19 và đã qua 14 ngày tính từ mũi tiêm cuối hoặc được công bố khỏi bệnh COVID-19 theo quy định. Cơ sở lưu trú được tổ chức ăn uống tại chỗ nhưng chỉ phục vụ khách hiện đang lưu trú; nhân viên phục vụ phải có xác nhận tiêm ít nhất 1 mũi vaccine phòng COVID-19 trở lên, định kỳ xét nghiệm SARS-CoV-2 cho nhân viên 1 tuần/lần…. Trao đổi với Lao Động, ông Phạm Hoàng Tuấn - Phó Chủ tịch UBND quận Đồ Sơn cho biết, việc thàn

## Load model

In [5]:
model_name = "unsloth/Phi-3-mini-4k-instruct"
adapter_id = "Hoang42/Phi3-4B-QLoRa-summary-adapter"
max_seq_length = 2048 # unsloth auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

In [6]:
def get_model_tokenizer(model_name, max_seq_length = 2048, dtype=None, load_in_4bit=True):
  model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
  )

  return model, tokenizer


In [7]:
bnb_4bit_compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8 else torch.float16

model, tokenizer = get_model_tokenizer(model_name, max_seq_length, bnb_4bit_compute_dtype, load_in_4bit)

==((====))==  Unsloth 2025.12.5: Fast Mistral patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((3072,), eps=1e-05)
        (post_attention_laye

## Base model Prediction

In [9]:
def summarize(text, prompt, model, tokenizer,
              max_new_tokens=1000, temperature=0.2):

  full_prompt = f"""<|user|>
  {prompt}

  Văn bản: {text}
  <|end|>
  <|assistant|>
  """

  inputs = tokenizer(
    full_prompt,
    return_tensors="pt",
    truncation=True, # Thêm tham số này
    max_length=model.max_seq_length
  ).to(model.device)

  generated_ids = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      temperature=temperature,
      do_sample=False,  # để chính xác, không ngẫu nhiên
      pad_token_id=tokenizer.eos_token_id,
  )

  # 4. Decode output
  decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=False)

  assistant_tag = "<|assistant|>"
  if assistant_tag in decoded:
      summary = decoded.split(assistant_tag, 1)[1]
  else:
      summary = decoded

  summary = summary.replace("<|end|>", "").strip()

  return summary


In [10]:
text = """
Tiêu đề: Đảm bảo phúc lợi cho người lao động trong dịch bệnh COVID-19

Nội dung: Buổi toạ đàm là dịp để các Công đoàn cơ sở tại doanh nghiệp được chia sẻ, trao đổi những kinh nghiệm thực tế, khó khăn và nỗ lực, từ đó rút ra bài học trong quá trình hoạt động ở cơ sở. Tại buổi tọa đàm, các đại biểu đã trao đổi nhiều kinh nghiệm về vận động doanh nghiệp duy trì, đảm bảo những điều khoản phúc lợi trong thỏa ước lao động tập thể; giải quyết chế độ, chính sách theo luật quy định của người lao động khi doanh nghiệp thu hẹp sản xuất, cắt giảm lao động. Đồng thời, đối thoại với chủ sử dụng lao động nhằm duy trì việc làm, thu nhập của người lao động trong bối cảnh dịch bệnh diễn biến phức tạp thời gian qua. Bà Đoàn Thị Kim Loan – Công đoàn cơ sở Công ty Changshin Việt Nam (xã Thạnh Phú, huyện Vĩnh Cửu, Đồng Nai) cho biết: Trong thời gian dịch bệnh, Công đoàn công ty đã có nhiều sáng kiến cùng Ban giám đốc công ty đảm bảo được việc phòng chống dịch bệnh, đồng thời đảm bảo được quyền lợi đầy đủ cho người lao động. Do đó, người lao động của công ty không bị nghỉ việc trong dịch COVID-19. Theo LĐLĐ tỉnh, từ đầu năm đến nay do ảnh hưởng của dịch bệnh COVID-19, toàn tỉnh có 129 doanh nghiệp và trên 100.000 người lao động gặp khó khăn, trong đó có trên 82.000 người lao động bị giảm giờ làm trong tuần hoặc nghỉ không hưởng lương, chấm dứt hợp đồng lao động. Trong đó, các nhóm ngành nghề ảnh hưởng nhiều nhất như: dệt may, giày da, sản xuất đồ gỗ, điện tử… Với những khó khăn nói trên, các cấp công đoàn đã có nhiều nỗ lực nhằm tuyên truyền, hướng dẫn, đối thoại, chia sẻ thông tin với doanh nghiệp để ổn định sản xuất, đảm bảo việc làm và các chế độ chính sách cho người lao động.
"""

prompt = "Hãy tóm tắt bài báo này sao cho tổng quát hết được nội dung"

summary = summarize(text, prompt, model, tokenizer, 1000)
print(summary)

Bài học trong buổi tọa đàm, các đạo đức đã trao đổi nhiều kinh nghiệm về vận động doanh nghiệp, đảm bảo những điều khoản phúc lợi trong thỏa ước lao động tập thể; giải quyết chế độ, chính sách theo luật quy định của người lao động khi doanh nghiệp thu hẹp sản xuất, cắt giảm lao động. Đồng thời, đối thoại với chủ sử dụng lao động nhằm duy trì việc làm, thu nhập của người lao động trong bối cảnh dịch bệnh diễn biến phức tạp thời gian qua. Bà Đoàn Thị Kim Loan – Công đoàn cơ sở Công ty Changshin Việt Nam (xã Thạnh Phú, huyện Vĩnh Cửu, Đồng Nai) cho biết: Trong thời gian dịch bệnh, Công đoàn công ty đã có nhiều sáng kiến cùng Ban giám đốc công ty đảm bảo được việc phòng chống dịch bệnh, đồng thời đảm bảo được quyền lợi đầy đủ cho người lao động. Do đó, người lao động của công ty không bị nghỉ việc trong dịch COVID-19. Theo LĐLĐ tỉnh, tỉnh đã có 129 doanh nghiệp và trên 100.000 người lao động gặp khó khăn, trong đó có trên 82.000 người lao động bị giảm giờ làm trong tuần hoặc nghỉ không hưở

In [11]:
from tqdm import tqdm # Process Bar

pred, refer = [], []
for sample in tqdm(new_eval, desc="Đánh giá mô hình"):
  pred.append(summarize(sample['input'], prompt, model, tokenizer, 1000))
  refer.append(sample['output'])

Đánh giá mô hình: 100%|██████████| 50/50 [30:20<00:00, 36.40s/it]


In [12]:
result_infer = {
  "predictions": pred,
  "references": refer,
}

output_file_path = 'base_model_inference.json'
with open(output_file_path, 'w', encoding='utf-8') as f:
  json.dump(result_infer, f, ensure_ascii=False, indent=4)


In [13]:
#@title Merge apdapter to base model
model_merged = PeftModel.from_pretrained(model, adapter_id)
model_merged.eval()

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/59.8M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj

## Fine tuned Prediction

In [14]:
text = """
Tiêu đề: Đảm bảo phúc lợi cho người lao động trong dịch bệnh COVID-19

Nội dung: Buổi toạ đàm là dịp để các Công đoàn cơ sở tại doanh nghiệp được chia sẻ, trao đổi những kinh nghiệm thực tế, khó khăn và nỗ lực, từ đó rút ra bài học trong quá trình hoạt động ở cơ sở. Tại buổi tọa đàm, các đại biểu đã trao đổi nhiều kinh nghiệm về vận động doanh nghiệp duy trì, đảm bảo những điều khoản phúc lợi trong thỏa ước lao động tập thể; giải quyết chế độ, chính sách theo luật quy định của người lao động khi doanh nghiệp thu hẹp sản xuất, cắt giảm lao động. Đồng thời, đối thoại với chủ sử dụng lao động nhằm duy trì việc làm, thu nhập của người lao động trong bối cảnh dịch bệnh diễn biến phức tạp thời gian qua. Bà Đoàn Thị Kim Loan – Công đoàn cơ sở Công ty Changshin Việt Nam (xã Thạnh Phú, huyện Vĩnh Cửu, Đồng Nai) cho biết: Trong thời gian dịch bệnh, Công đoàn công ty đã có nhiều sáng kiến cùng Ban giám đốc công ty đảm bảo được việc phòng chống dịch bệnh, đồng thời đảm bảo được quyền lợi đầy đủ cho người lao động. Do đó, người lao động của công ty không bị nghỉ việc trong dịch COVID-19. Theo LĐLĐ tỉnh, từ đầu năm đến nay do ảnh hưởng của dịch bệnh COVID-19, toàn tỉnh có 129 doanh nghiệp và trên 100.000 người lao động gặp khó khăn, trong đó có trên 82.000 người lao động bị giảm giờ làm trong tuần hoặc nghỉ không hưởng lương, chấm dứt hợp đồng lao động. Trong đó, các nhóm ngành nghề ảnh hưởng nhiều nhất như: dệt may, giày da, sản xuất đồ gỗ, điện tử… Với những khó khăn nói trên, các cấp công đoàn đã có nhiều nỗ lực nhằm tuyên truyền, hướng dẫn, đối thoại, chia sẻ thông tin với doanh nghiệp để ổn định sản xuất, đảm bảo việc làm và các chế độ chính sách cho người lao động.
"""

prompt = "Hãy tóm tắt bài báo này sao cho tổng quát hết được nội dung"

summary = summarize(text, prompt, model_merged, tokenizer, 1000)
print(summary)

Công đoàn cơ sở Công ty Changshin Việt Nam (xã Thạnh Phú, huyện Vĩnh Cửu, Đồng Nai) đã tổ chức buổi tọa đàm “Tạo điều kiện cho người lao động trong dịch bệnh COVID-19” với sự tham gia của 10 đại biểu Công đoàn cơ sở trực thuộc.


In [15]:
new_eval[9]['input']

'Tiêu đề: Ngọc Sơn đã liên hệ được với Hồ Văn Cường và mong được gặp để giúp đỡ\n\nNội dung: Thời gian qua, thông tin Hồ Văn Cường nhận được sự quan tâm của đông đảo người hâm mộ. Bởi sau khi cố ca sĩ Phi Nhung qua đời, nam ca sĩ trở thành tâm điểm khi được quản lý cố ca sĩ trả tiền cát - xê. Sau đó, anh chia sẻ sẽ dọn ra khỏi nhà quản lý cố ca sĩ Phi Nhung và tập trung cho việc học. Kể từ khi có thông tin Hồ Văn Cường quyết định rời khỏi công ty của Phi Nhung, một số khán giả vẫn rất lo lắng cho gia đình nam ca sĩ. Mọi người cũng ngóng chờ phía Hồ Văn Cường cập nhật thêm thông tin về cuộc sống mới. Đặc biệt, có một số khán giả hoài nghi, liệu rằng Hồ Văn Cường đã dọn ra ở riêng hay vẫn ở nhà, chịu sự giám sát của quản lý cố ca sĩ Phi Nhung? Trước đó, đã có một số người nổi tiếng, những doanh nhân (trong đó có danh ca Ngọc Sơn) muốn đứng ra hỗ trợ gia đình Hồ Văn Cường nhưng không ai trong số này liên lạc được với nam ca sĩ. Mới đây, danh ca Ngọc Sơn chia sẻ trên trang cá nhân: "Chúng 

In [16]:
summary = summarize(new_eval[5]['input'], prompt, model_merged, tokenizer, 1000)
print(summary)

Một cây xăng ở Xuân Lộc, Đồng Nai bốc cháy, người dân hoảng loạn.


In [17]:
predictions, references = [], []

for sample in tqdm(new_eval, desc="Đánh giá mô hình"):
  predictions.append(summarize(sample['input'], prompt, model_merged, tokenizer, 1000))
  references.append(sample['output'])

Đánh giá mô hình: 100%|██████████| 50/50 [13:08<00:00, 15.77s/it]


In [18]:
# Tạo File inference và lưu ra file JSON
result = {
  "predictions": predictions,
  "references": references,
}

output_file_path = "fine_tuned_inference.json"
with open(output_file_path, "w", encoding="utf-8") as f:
  json.dump(result, f, ensure_ascii=False, indent=4)

##CPU core

In [19]:
%%capture
!pip install evaluate rouge-score bert-score

import json
from evaluate import load
from bert_score import score
from prettytable import PrettyTable

In [20]:
with open('base_model_inference.json', 'r', encoding='utf-8') as f:
  data = json.load(f)
predictions_b = data['predictions']
references_b = data['references']


with open("fine_tuned_inference.json", "r", encoding="utf-8") as f:
  data = json.load(f)
predictions_f = data["predictions"]
references_f = data["references"]

In [21]:
#@title ROUGE scores
rouge = load("rouge")

rouge_b = rouge.compute(
  predictions=predictions_b,
  references=references_b,
  use_stemmer=True
)

rouge_f = rouge.compute(
  predictions=predictions_f,
  references=references_f,
  use_stemmer=True
)


rouge_b

{'rouge1': np.float64(0.2017844052728299),
 'rouge2': np.float64(0.1104835815699289),
 'rougeL': np.float64(0.15357141600369306),
 'rougeLsum': np.float64(0.15685973702856976)}

In [22]:
table = PrettyTable()
table.field_names = ["Model", "Rouge1", "Rouge2", "RougeL", 'RougeLsum']
table.add_row( [
  'Base model',
  f'{rouge_b["rouge1"]:.4f}',
  f'{rouge_b["rouge2"]:.4f}',
  f'{rouge_b["rougeL"]:.4f}',
  f'{rouge_b['rougeLsum']:.4f}'
])

table.add_row([
  'Fine tuned',
  f'{rouge_f["rouge1"]:.4f}',
  f'{rouge_f["rouge2"]:.4f}',
  f'{rouge_f["rougeL"]:.4f}',
  f'{rouge_f['rougeLsum']:.4f}',
])

print(table)

+------------+--------+--------+--------+-----------+
|   Model    | Rouge1 | Rouge2 | RougeL | RougeLsum |
+------------+--------+--------+--------+-----------+
| Base model | 0.2018 | 0.1105 | 0.1536 |   0.1569  |
| Fine tuned | 0.4942 | 0.2778 | 0.3443 |   0.3451  |
+------------+--------+--------+--------+-----------+


In [23]:
#@title BERT Score
table = PrettyTable()

table.field_names = ["Model", "P", "R", "F1"]

P, R, F1 = score(predictions_b, references_b, lang="vi", device='cpu', batch_size=4)
table.add_row(['Base model', f'{P.mean():.4f}', f'{R.mean():.4f}', f'{F1.mean():.4f}'])

P, R, F1 = score(predictions_f, references_f, lang="vi", device='cpu', batch_size=4)
table.add_row(['Fine tuned', f'{P.mean():.4f}', f'{R.mean():.4f}', f'{F1.mean():.4f}'])

print(table)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

+------------+--------+--------+--------+
|   Model    |   P    |   R    |   F1   |
+------------+--------+--------+--------+
| Base model | 0.6010 | 0.7124 | 0.6512 |
| Fine tuned | 0.7563 | 0.7528 | 0.7522 |
+------------+--------+--------+--------+
